In [1]:
 """Session 26: real PointRend R101, group-aware five-fold OOF.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import signal
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import urlopen

BUILD = 'SESSION26-POINTREND-R101-V7-BATCH-BUDGET'
CLASSES = ['Nail_psoriasis', 'SJS-TEN', 'acne', 'vitiligo']
EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
ENV_ROOT = Path('/tmp/session26_openmmlab')
ENV_PYTHON = ENV_ROOT / 'venv/bin/python'
MMCV_WHEEL = ('https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/'
              'mmcv-2.1.0-cp311-cp311-manylinux1_x86_64.whl')
R101_WEIGHTS = ('https://download.openmmlab.com/mmdetection/v2.0/mask_rcnn/'
    'mask_rcnn_r101_caffe_fpn_mstrain-poly_3x_coco/'
    'mask_rcnn_r101_caffe_fpn_mstrain-poly_3x_coco_20210526_132339-3c33ce02.pth')
POINT_WEIGHTS = ('https://download.openmmlab.com/mmdetection/v2.0/point_rend/'
    'point_rend_r50_caffe_fpn_mstrain_3x_coco/'
    'point_rend_r50_caffe_fpn_mstrain_3x_coco-e0ebb6b7.pth')
BASE_CONFIG = 'mmdet::point_rend/point-rend_r50-caffe_fpn_ms-1x_coco.py'
AUGMENT_PREFIX = re.compile(
    r'^(?:Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|'
    r'padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|Resize-450_400|'
    r'flip-(?:-1|0|1)|(?:hue|saturation)-(?:200|150|100|50)|'
    r'superpixels-(?:200|100|75|50|25))')


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def headless_environment(environment=None):
    """Do not inherit Kaggle's inline backend into a private non-notebook Python.

    Return a copy so the interactive notebook keeps its own plotting settings.
    Use assignment, not setdefault: Kaggle already defines MPLBACKEND.
    """
    env = dict(os.environ if environment is None else environment)
    env['MPLBACKEND'] = 'Agg'
    env['MPLCONFIGDIR'] = str(ENV_ROOT / 'matplotlib')
    return env


def run_command(command, timeout=None, environment=None, log=None):
    """Bound the whole child process group, including torchrun workers."""
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    handle = Path(log).open('w') if log else None
    try:
        process = subprocess.Popen(command, env=headless_environment(environment), start_new_session=True,
                                   stdout=handle, stderr=subprocess.STDOUT if handle else None)
        started = time.monotonic()
        offset = 0
        while process.poll() is None:
            if timeout is not None and time.monotonic() - started > timeout:
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                raise TimeoutError(f'Time budget exceeded for {command[:5]}; log={log}')
            if log:
                with Path(log).open() as reader:
                    reader.seek(offset)
                    content = reader.read()
                    offset = reader.tell()
                if content:
                    print(content, end='', flush=True)
            time.sleep(1)
        if log:
            with Path(log).open() as reader:
                reader.seek(offset)
                print(reader.read(), end='', flush=True)
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if handle:
            handle.close()


PROBE = r'''
import sys, numpy, torch, torchvision, mmcv, mmengine, mmdet
from mmcv.ops import nms, roi_align, point_sample
from mmdet.models import PointRend
assert sys.version_info[:2] == (3, 11)
assert torch.__version__.split('+')[0] == '2.1.0'
assert torchvision.__version__.split('+')[0] == '0.16.0'
assert torch.version.cuda == '11.8'
assert numpy.__version__ == '1.26.4'
assert mmcv.__version__ == '2.1.0'
assert mmengine.__version__ == '0.10.7'
assert mmdet.__version__ == '3.3.0'
assert torch.cuda.is_available(), 'Select GPU T4 x2 in Kaggle settings'
x = torch.randn(1, 4, 16, 16, device='cuda', requires_grad=True)
rois = torch.tensor([[0., 0., 0., 12., 12.]], device='cuda')
y = roi_align(x, rois, (7, 7), 1., 0, 'avg', True)
p = point_sample(x, torch.rand(1, 16, 2, device='cuda'))
(y.sum() + p.sum()).backward()
assert x.grad is not None and torch.isfinite(x.grad).all()
boxes = rois[:, 1:]
nms(boxes, torch.ones(1, device='cuda'), 0.5)
print('SUPPORTED ENVIRONMENT OK', sys.executable, torch.__version__,
      mmcv.__version__, mmdet.__version__, 'GPUs=', torch.cuda.device_count())
'''


def bootstrap(arguments, started):
    """Python 3.11 avoids the unsupported MMCV 2.2 version-guard workaround."""
    if os.environ.get('SESSION26_ISOLATED') == '1':
        os.environ.update(headless_environment())
        return None
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Training bootstrap is for Kaggle Linux GPUs. Use --audit-only locally.')
    smi = shutil.which('nvidia-smi')
    if not smi:
        raise RuntimeError('Enable GPU T4 x2 before running Session 26.')
    run_command([smi, '--query-gpu=name,memory.total', '--format=csv,noheader'], timeout=30)
    if shutil.disk_usage('/tmp').free < 9 * 1024**3:
        raise RuntimeError('Need 9 GiB of temporary disk space for Python/Torch. Start a fresh session.')
    env = headless_environment()
    env.pop('PYTHONPATH', None)
    env.pop('PYTHONHOME', None)
    env.update(PYTHONNOUSERSITE='1', PYTHONUNBUFFERED='1',
               UV_PYTHON_INSTALL_DIR=str(ENV_ROOT / 'python'),
               UV_CACHE_DIR=str(ENV_ROOT / 'cache'),
               UV_PYTHON_BIN_DIR=str(ENV_ROOT / 'bin'))
    ok = False
    if ENV_PYTHON.is_file():
        check = subprocess.run([str(ENV_PYTHON), '-c', PROBE], env=env,
                               capture_output=True, text=True, timeout=120)
        ok = check.returncode == 0
        print(check.stdout + check.stderr, flush=True)
    if not ok:
        # A private uv installation: no changes to Kaggle's notebook packages.
        installer = ENV_ROOT / 'installer'
        run_command([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                     '--target', installer, 'uv==0.8.22'], timeout=300, environment=env)
        uv = installer / 'bin/uv'
        run_command([uv, 'venv', '--python', '3.11', '--seed', ENV_ROOT / 'venv',
                     '--allow-existing'], timeout=420, environment=env)
        pip = [ENV_PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '--timeout', '90']
        run_command(pip + ['pip==25.2', 'setuptools==75.1.0', 'wheel==0.44.0'],
                    timeout=300, environment=env)
        run_command(pip + ['torch==2.1.0', 'torchvision==0.16.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu118'],
                    timeout=1500, environment=env)
        run_command(pip + ['numpy==1.26.4', 'scipy==1.14.1', 'Pillow==10.4.0',
                    'opencv-python==4.10.0.84', 'matplotlib==3.9.2', 'PyYAML==6.0.2',
                    'pycocotools==2.0.10', 'shapely==2.0.6', 'six==1.16.0',
                    'terminaltables==3.1.10', 'tqdm==4.66.5', 'addict==2.4.0',
                    'termcolor==2.4.0', 'packaging==24.1', 'rich==13.9.4',
                    'yapf==0.40.2', 'mmengine==0.10.7'], timeout=600, environment=env)
        run_command(pip + ['--no-deps', MMCV_WHEEL, 'mmdet==3.3.0'],
                    timeout=600, environment=env)
        run_command([ENV_PYTHON, '-m', 'pip', 'check'], timeout=60, environment=env)
        run_command([ENV_PYTHON, '-c', PROBE], timeout=120, environment=env)
    env.update(SESSION26_ISOLATED='1', SESSION26_STARTED=str(started))
    path = globals().get('__file__')
    if not path or not Path(path).is_file():
        # Supports pasting the complete standalone source into one Kaggle cell.
        shell = get_ipython()  # noqa: F821
        source = next(text for text in reversed(shell.user_ns['In'])
                      if BUILD in text and 'def bootstrap(' in text)
        path = '/kaggle/working/SESSION_26_runtime.py'
        Path(path).write_text(source)
    run_command([ENV_PYTHON, '-u', path, *arguments], environment=env,
                timeout=max(1, 11.6 * 3600 - (time.time() - started)))
    return 0


def split_dirs(root, split):
    aliases = ('valid', 'val') if split == 'valid' else (split,)
    for alias in aliases:
        for image, label in ((root / alias / 'images', root / alias / 'labels'),
                             (root / 'images' / alias, root / 'labels' / alias)):
            if image.is_dir() and label.is_dir():
                return image, label
    return None


def dataset_image_counts(root):
    return tuple(sum(p.suffix.lower() in EXTENSIONS for p in split_dirs(root, s)[0].iterdir())
                 if split_dirs(root, s) else 0 for s in ('train', 'valid', 'test'))


def dataset_priority(root):
    counts = dataset_image_counts(root)
    return (int(counts == (6057, 1085, 1030)),
            int(root.name == 'Image_Annotation_rebuilt'),
            int((root / 'metadata/manifest.csv').is_file()),
            int((root / 'cv/fold_0/train.txt').is_file()))


def detect_dataset(hint='auto'):
    import yaml
    search = Path('/kaggle/input') if Path('/kaggle/input').is_dir() else Path.cwd()
    candidates = [Path(hint).expanduser()] if hint != 'auto' else sorted(
        {p.parent for p in search.rglob('data.yaml')
         if not any(part in {'removed_leakage', 'cv', '.git'} for part in p.parts)})
    found = []
    for root in candidates:
        if not (root / 'data.yaml').is_file() or not all(
                split_dirs(root, s) for s in ('train', 'valid')):
            continue
        doc = yaml.safe_load((root / 'data.yaml').read_text())
        names = doc.get('names', [])
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names, key=int)]
        if names != CLASSES:
            continue
        # This session is requested for the 6057/1085/1030 rebuilt export.
        # Kaggle uploads may also contain an older Image_Annotation tree and a
        # "New Folder With Items" copy. Prefer the exact named target when
        # several valid four-class roots are attached; never trust data.yaml's
        # stale absolute `path:` field for selecting the image directories.
        score = dataset_priority(root)
        found.append((score, root.resolve()))
    if not found:
        raise FileNotFoundError('Attach the four-class YOLO polygon dataset (train/valid images and labels).')
    found.sort(key=lambda pair: (pair[0], str(pair[1])), reverse=True)
    best = [root for score, root in found if score == found[0][0]]
    if len(best) > 1:
        raise RuntimeError('Multiple equally suitable datasets; set DATA_ROOT: ' + ', '.join(map(str, best)))
    print('[data] selected', best[0], 'counts=', dataset_image_counts(best[0]), flush=True)
    return best[0]


def read_items(root):
    from PIL import Image
    manifest = root / 'metadata/manifest.csv'
    metadata = {}
    if manifest.is_file():
        with manifest.open() as f:
            for row in csv.DictReader(f):
                key = Path(row['image_file']).name
                if key in metadata:
                    raise ValueError(f'Duplicate manifest basename: {key}')
                metadata[key] = row
    items = []
    for split in ('train', 'valid', 'test'):
        pair = split_dirs(root, split)
        if not pair:
            continue
        for image in sorted(pair[0].iterdir()):
            if image.suffix.lower() not in EXTENSIONS:
                continue
            label = pair[1] / (image.stem + '.txt')
            if not label.is_file():
                raise FileNotFoundError(f'Missing label: {label}')
            with Image.open(image) as handle:
                rgb = handle.convert('RGB')
                width, height = rgb.size
                pixel = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            polygons = []
            for line_number, line in enumerate(label.read_text().splitlines(), 1):
                if not line.strip():
                    continue
                values = line.split()
                if len(values) < 7 or (len(values) - 1) % 2:
                    raise ValueError(f'Expected polygon, not box: {label}:{line_number}')
                cls = int(values[0])
                xy = list(map(float, values[1:]))
                if cls not in range(4) or not all(math.isfinite(v) and 0 <= v <= 1 for v in xy):
                    raise ValueError(f'Invalid class/coordinates: {label}:{line_number}')
                x, y = xy[0::2], xy[1::2]
                area = abs(sum(x[i] * y[(i+1) % len(x)] - x[(i+1) % len(x)] * y[i]
                               for i in range(len(x)))) / 2 * width * height
                if area <= 1 or max(x) <= min(x) or max(y) <= min(y):
                    raise ValueError(f'Degenerate polygon: {label}:{line_number}')
                polygons.append((cls, xy, area))
            row = metadata.get(image.name, {})
            if metadata and not row.get('split_group_id'):
                raise ValueError(f'Missing split_group_id in manifest for {image}')
            source = AUGMENT_PREFIX.sub('', image.stem.split('.rf.', 1)[0], count=1)
            items.append(dict(id=len(items)+1, image=str(image.resolve()),
                relative=str(image.relative_to(root)), split=split, width=width, height=height,
                polygons=polygons, classes=sorted({p[0] for p in polygons}), pixel=pixel,
                source=row.get('split_group_id') or source, manifest=bool(metadata),
                declared_fold=row.get('cv_fold', ''),
                label_sha256=hashlib.sha256(label.read_bytes()).hexdigest()))
    if not items:
        raise ValueError('No images in dataset')
    # Union source families and exact decoded-pixel duplicates, including cases
    # whose export names differ. No inference of patient identity is made.
    parent = list(range(len(items)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    seen = {}
    for i, item in enumerate(items):
        for token in (('source', item['source']), ('pixel', item['pixel'])):
            if token in seen:
                parent[find(i)] = find(seen[token])
            else:
                seen[token] = i
    groups = defaultdict(list)
    for i, item in enumerate(items):
        groups[find(i)].append(item)
    for members in groups.values():
        gid = fingerprint(sorted({m['source'] for m in members}))[:20]
        for item in members:
            item['group'] = gid
    return items


def make_folds(items, seed):
    import numpy as np
    development = [x for x in items if x['split'] != 'test']
    groups = defaultdict(list)
    for item in development:
        groups[item['group']].append(item)
    if len(groups) < 5:
        raise ValueError('Five-fold CV requires at least five independent groups.')
    units = sorted(groups)
    features = np.array([[sum(c in x['classes'] for x in groups[g]) for c in range(4)]
                         + [len(groups[g])] for g in units], dtype=float)
    assignment = {}
    use_manifest = all(x['manifest'] and x['declared_fold'] in {'0','1','2','3','4'}
                       for x in development)
    if use_manifest:
        for g, members in groups.items():
            values = {int(x['declared_fold']) for x in members}
            if len(values) != 1:
                raise ValueError('Manifest separates related/duplicate images across CV folds.')
            assignment[g] = values.pop()
    else:
        # Search deterministic group allocations; never move individual images.
        target = np.maximum(features.sum(axis=0) / 5, 1)
        best = None
        for trial in range(32):
            rng = random.Random(seed + trial)
            order = list(range(len(units)))
            rng.shuffle(order)
            order.sort(key=lambda i: -float((features[i] / target).max()))
            counts = np.zeros((5, 5))
            current = {}
            for i in order:
                scores = []
                for f in range(5):
                    after = counts.copy()
                    after[f] += features[i]
                    scores.append(float(((after / target) ** 2).sum()))
                lowest = min(scores)
                choices = [f for f in range(5) if abs(scores[f] - lowest) < 1e-10]
                chosen = rng.choice(choices)
                counts[chosen] += features[i]
                current[units[i]] = chosen
            objective = int((counts[:, :4] == 0).sum()) * 1e6 + float(
                (((counts - target) / target) ** 2).sum())
            if best is None or objective < best[0]:
                best = objective, current
        assignment = best[1]
    folds = []
    test_groups = {x['group'] for x in items if x['split'] == 'test'}
    overlap = test_groups & set(groups)
    if use_manifest and overlap:
        raise ValueError('Group or pixel leakage with the V2 locked test set.')
    for f in range(5):
        train = [x for x in development if assignment[x['group']] != f]
        valid = [x for x in development if assignment[x['group']] == f]
        assert not {x['group'] for x in train} & {x['group'] for x in valid}
        assert not {x['pixel'] for x in train} & {x['pixel'] for x in valid}
        for subset in (train, valid):
            if set(c for x in subset for c in x['classes']) != set(range(4)):
                raise ValueError(f'Fold {f} lacks a class; supply reviewed group-balanced manifests.')
        folds.append(dict(fold=f, train=train, valid=valid))
    assert sorted(x['id'] for fold in folds for x in fold['valid']) == sorted(
        x['id'] for x in development)
    report = dict(development_images=len(development), independent_groups=len(groups),
        split_counts=dict(Counter(x['split'] for x in items)),
        mode='V2 declared group folds' if use_manifest else 'inferred source + exact-pixel grouped CV',
        train_valid_group_overlap=0, train_valid_pixel_overlap=0,
        test_group_overlap=len(overlap), test_evaluated=False,
        patient_independence_verified=False,
        annotation_vertices=dict(Counter(str(len(p[1]) // 2) for x in development for p in x['polygons'])),
        note='Legacy test source relatives remain in development; no independent test claim.' if overlap
             else 'No detected group overlap with test; patient identities may be unknown.')
    return folds, report


def coco_document(items):
    images, annotations = [], []
    for item in items:
        images.append(dict(id=item['id'], file_name=item['image'],
                           width=item['width'], height=item['height']))
        for cls, xy, area in item['polygons']:
            xs = [v * item['width'] for v in xy[0::2]]
            ys = [v * item['height'] for v in xy[1::2]]
            annotations.append(dict(id=len(annotations)+1, image_id=item['id'],
                category_id=cls+1, segmentation=[[v for pair in zip(xs, ys) for v in pair]],
                bbox=[min(xs), min(ys), max(xs)-min(xs), max(ys)-min(ys)], area=area, iscrowd=0))
    return dict(info={'description': BUILD}, licenses=[], images=images, annotations=annotations,
                categories=[dict(id=i+1, name=name) for i, name in enumerate(CLASSES)])


def download(url, destination):
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size > 1024**2:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            partial = destination.with_suffix('.partial')
            with urlopen(url, timeout=90) as response, partial.open('wb') as output:
                shutil.copyfileobj(response, output)
            if partial.stat().st_size < 1024**2:
                raise IOError('Checkpoint is unexpectedly small')
            partial.replace(destination)
            return destination
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2)


def register_runtime():
    import torch
    from mmengine.hooks import Hook
    from mmdet.registry import MODELS, HOOKS
    from mmdet.utils import register_all_modules
    from mmdet.models.roi_heads import PointRendRoIHead
    register_all_modules(init_default_scope=True)

    @MODELS.register_module(force=True)
    class SafePointRendRoIHead(PointRendRoIHead):
        def _mask_point_loss(self, x, sampling_results, mask_preds, batch_gt_instances):
            if sum(len(r.pos_bboxes) for r in sampling_results) == 0:
                zero = mask_preds.sum() * 0
                for parameter in self.point_head.parameters():
                    zero = zero + parameter.sum() * 0
                return dict(loss_point=zero)
            return super()._mask_point_loss(x, sampling_results, mask_preds, batch_gt_instances)

    @HOOKS.register_module(force=True)
    class WallClockHook(Hook):
        priority = 'NORMAL'
        def __init__(self, seconds, lr):
            self.seconds, self.lr = seconds, lr
        def before_train(self, runner):
            self.started = time.monotonic()
            self.last_step = self.started
            runner.session26_step_times = []
        def before_train_iter(self, runner, batch_idx, data_batch=None):
            fraction = max(runner.iter / runner.max_iters,
                           (time.monotonic() - self.started) / self.seconds)
            cosine = 0.01 + 0.99 * (1 + math.cos(math.pi * min(fraction, 1))) / 2
            warmup = min(1., (runner.iter + 1) / min(250, max(1, runner.max_iters // 10)))
            for group in runner.optim_wrapper.optimizer.param_groups:
                group['lr'] = self.lr * cosine * warmup
        def after_train_iter(self, runner, batch_idx, data_batch=None, outputs=None):
            now = time.monotonic()
            runner.session26_step_times.append(now - self.last_step)
            self.last_step = now
            for name, value in (outputs or {}).items():
                if 'loss' in name and not torch.isfinite(torch.as_tensor(value)).all():
                    raise FloatingPointError(f'Nonfinite training loss: {name}={value}')
            if time.monotonic() - self.started >= self.seconds:
                runner.train_loop.stop_training = True
        def before_test(self, runner):
            runner.session26_prediction_times = []
        def before_test_iter(self, runner, batch_idx, data_batch=None):
            torch.cuda.synchronize()
            self.prediction_started = time.monotonic()
        def after_test_iter(self, runner, batch_idx, data_batch=None, outputs=None):
            torch.cuda.synchronize()
            runner.session26_prediction_times.append(time.monotonic() - self.prediction_started)


def build_config(job):
    from mmengine.config import Config
    import mmdet
    # The package:: syntax is only supported inside _base_, not fromfile().
    base = Path(mmdet.__file__).parent / '.mim/configs' / BASE_CONFIG.split('::', 1)[1]
    if not base.is_file():
        raise FileNotFoundError(f'MMDetection wheel is missing its packaged configuration: {base}')
    cfg = Config.fromfile(str(base))
    cfg.model.backbone.update(depth=101, init_cfg=None, frozen_stages=1)
    cfg.model.roi_head.type = 'SafePointRendRoIHead'
    cfg.model.roi_head.bbox_head.num_classes = 4
    cfg.model.roi_head.mask_head.num_classes = 4
    cfg.model.roi_head.point_head.num_classes = 4
    size = job['size']
    train_pipeline = [dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(type='RandomResize', scale=(size, size), ratio_range=(0.8, 1.2), keep_ratio=True),
        dict(type='RandomFlip', prob=0.5), dict(type='PackDetInputs')]
    test_pipeline = [dict(type='LoadImageFromFile'),
        dict(type='Resize', scale=(size, size), keep_ratio=True),
        dict(type='PackDetInputs', meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor'))]
    batch_size = job.get('batch_size', 1)
    learning_rate = 0.00125 * batch_size
    def loader(annotation, training):
        return dict(batch_size=batch_size if training else 1, num_workers=2, persistent_workers=True,
            pin_memory=True, sampler=dict(type='InfiniteSampler' if training else 'DefaultSampler',
                                         shuffle=training),
            dataset=dict(type='CocoDataset', data_root='', ann_file=annotation,
                data_prefix=dict(img=''), metainfo=dict(classes=tuple(CLASSES)),
                filter_cfg=dict(filter_empty_gt=True, min_size=16) if training else None,
                test_mode=not training, pipeline=train_pipeline if training else test_pipeline))
    cfg.train_dataloader = loader(job['train_json'], True)
    cfg.test_dataloader = loader(job['valid_json'], False)
    cfg.val_dataloader = cfg.val_evaluator = cfg.val_cfg = None
    cfg.train_cfg = dict(type='IterBasedTrainLoop', max_iters=job['max_iters'], val_interval=10**9)
    cfg.test_cfg = dict(type='TestLoop', fp16=True)
    cfg.test_evaluator = dict(type='CocoMetric', ann_file=job['valid_json'],
        metric=['bbox', 'segm'], format_only=True, outfile_prefix=str(Path(job['work']) / 'oof'))
    cfg.optim_wrapper = dict(type='AmpOptimWrapper', loss_scale=dict(init_scale=128., growth_interval=2000),
        optimizer=dict(type='SGD', lr=learning_rate, momentum=0.9, weight_decay=0.0001),
        clip_grad=dict(max_norm=10, norm_type=2))
    cfg.param_scheduler = []
    cfg.custom_hooks = [dict(type='WallClockHook', seconds=job['train_seconds'], lr=learning_rate)]
    cfg.default_hooks.checkpoint = dict(type='CheckpointHook', by_epoch=False,
        # The worker saves one final checkpoint after training. Frequent interim
        # multi-GB writes waste the two-GPU Kaggle time/disk budget and were not
        # resumable by this runner.
        interval=job['max_iters'] + 1, max_keep_ckpts=1, save_last=False, save_optimizer=False,
        save_param_scheduler=False)
    cfg.default_hooks.logger = dict(type='LoggerHook', interval=50, log_metric_by_epoch=False)
    cfg.log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
    cfg.visualizer = dict(type='DetLocalVisualizer', vis_backends=[], name='visualizer')
    cfg.env_cfg = dict(cudnn_benchmark=False, mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
                       dist_cfg=dict(backend='nccl'))
    cfg.randomness = dict(seed=job['seed'], deterministic=False)
    cfg.work_dir = job['work']
    cfg.load_from = job['initial']
    cfg.resume = False
    cfg.launcher = 'none'
    return cfg


def prepare_weights(job):
    import torch
    from mmdet.registry import MODELS
    cfg = build_config(job)
    model = MODELS.build(cfg.model)
    state = model.state_dict()
    combined, reports = {}, {}
    for name, url in [('r101', R101_WEIGHTS), ('point', POINT_WEIGHTS)]:
        path = download(url, ENV_ROOT / (name + '.pth'))
        raw = torch.load(path, map_location='cpu')
        raw = raw.get('state_dict', raw)
        accepted = {}
        for key, value in raw.items():
            key = key.removeprefix('module.')
            allowed = (not key.startswith(('roi_head.mask_head.', 'roi_head.point_head.'))
                       if name == 'r101' else key.startswith(('roi_head.mask_head.', 'roi_head.point_head.')))
            if allowed and key in state and value.shape == state[key].shape:
                accepted[key] = value
        if not accepted:
            raise RuntimeError(f'No compatible weights from {name}')
        reports[name] = dict(loaded_tensors=len(accepted), keys=sorted(accepted))
        combined.update(accepted)
        del raw
    if not any('backbone.layer3.22.' in key for key in combined):
        raise RuntimeError('R101 backbone checkpoint verification failed')
    torch.save(dict(state_dict=combined, meta=dict(build=BUILD)), job['initial'])
    save_json(Path(job['work']).parent / 'initialization_report.json', reports)


def worker(job):
    # Also protects a worker invoked directly, without bootstrap/run_command.
    os.environ.update(headless_environment())
    if job['action'] == 'evaluate':
        output = Path(job['output'])
        results = evaluate_oof(output, None, list(range(5)))
        save_json(output / 'evaluation_metrics.json', results)
        return
    import torch
    from mmengine.runner import Runner
    register_runtime()
    torch.set_num_threads(2)
    if job['action'] == 'prepare':
        prepare_weights(job)
        return
    cfg = build_config(job)
    cfg.dump(str(Path(job['work']) / 'resolved_config.py'))
    runner = Runner.from_cfg(cfg)
    from mmdet.models import PointRend
    from mmdet.models.roi_heads import PointRendRoIHead
    assert isinstance(runner.model, PointRend)
    assert isinstance(runner.model.roi_head, PointRendRoIHead)
    assert len(runner.model.backbone.layer3) == 23, 'Not a ResNet-101 backbone'
    gradient_checks = {'nonzero_point_gradients': 0, 'overflow_point_gradients': 0}
    def check_gradient(gradient):
        if runner.iter >= 2:
            return gradient  # Do not contaminate the steady-state timing benchmark.
        if not torch.isfinite(gradient).all():
            # AMP's scaler must be allowed to skip an overflowing step and adapt.
            gradient_checks['overflow_point_gradients'] += 1
        elif gradient.abs().max().item() > 0:
            gradient_checks['nonzero_point_gradients'] += 1
        return gradient
    handles = []
    if job['action'] == 'preflight':
        handles = [p.register_hook(check_gradient) for p in runner.model.roi_head.point_head.parameters()
                   if p.requires_grad]
    started = time.monotonic()
    runner.train()
    train_elapsed = time.monotonic() - started
    if runner.iter < 1:
        raise RuntimeError('No training iterations completed')
    runner.save_checkpoint(job['work'], 'final.pth', save_optimizer=False, save_param_scheduler=False,
                           meta=dict(dataset_meta=runner.train_dataloader.dataset.metainfo))
    started = time.monotonic()
    runner.test()
    evaluation_seconds = time.monotonic() - started
    if job['action'] == 'preflight':
        for handle in handles:
            handle.remove()
        if not gradient_checks['nonzero_point_gradients']:
            raise RuntimeError('Preflight did not exercise the PointRend training branch')
        save_json(Path(job['work']) / 'preflight.json', dict(gradient_checks,
            train_seconds=train_elapsed, evaluation_seconds=evaluation_seconds,
            seconds_per_iteration=sum(runner.session26_step_times[10:] or runner.session26_step_times) /
                                  len(runner.session26_step_times[10:] or runner.session26_step_times),
            prediction_seconds_per_image=sum(runner.session26_prediction_times) /
                                         len(runner.session26_prediction_times),
            batch_size=job.get('batch_size', 1),
            peak_gpu_gib=torch.cuda.max_memory_allocated() / 1024**3,
            peak_reserved_gib=torch.cuda.max_memory_reserved() / 1024**3,
            total_gpu_gib=torch.cuda.get_device_properties(0).total_memory / 1024**3))
        return
    # Only mark success after training, checkpoint saving AND held-out inference.
    save_json(Path(job['work']) / 'completed.json', dict(fold=job['fold'], signature=job['signature'],
        iterations=runner.iter, target_iterations=job['max_iters'],
        requested_iteration_cap=job['requested_max_iters'],
        allocated_target_reached=runner.iter >= job['max_iters'],
        requested_cap_reached=runner.iter >= job['requested_max_iters'],
        train_seconds=train_elapsed,
        evaluation_seconds=evaluation_seconds, validation_images=job['validation_images'],
        batch_size=job.get('batch_size', 1), learning_rate=0.00125 * job.get('batch_size', 1),
        training_images_seen=runner.iter * job.get('batch_size', 1),
        effective_epochs=runner.iter * job.get('batch_size', 1) / len(runner.train_dataloader.dataset),
        peak_gpu_gib=torch.cuda.max_memory_allocated() / 1024**3,
        budget_stopped=runner.iter < job['max_iters']))


def evaluate_oof(output, development, folds):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import numpy as np
    ground = output / 'oof_ground_truth.json'
    if development is not None:
        save_json(ground, coco_document(development))
    gt = COCO(str(ground))
    results = {}
    for metric in ('bbox', 'segm'):
        predictions = []
        for f in folds:
            predictions.extend(json.loads((output / f'fold_{f}' / f'oof.{metric}.json').read_text()))
        save_json(output / f'oof.{metric}.json', predictions)
        if metric == 'segm':
            predictions = [{k: v for k, v in p.items() if k != 'bbox'} for p in predictions]
        if predictions:
            dt = gt.loadRes(predictions)
        else:
            dt = COCO()
            dt.dataset = dict(images=gt.dataset['images'], categories=gt.dataset['categories'], annotations=[])
            dt.createIndex()
        evaluator = COCOeval(gt, dt, metric)
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
        precision = evaluator.eval['precision']
        per_class = {}
        for c, name in enumerate(CLASSES):
            values = precision[0, :, c, 0, -1]
            per_class[name] = float(np.mean(values[values >= 0])) if (values >= 0).any() else None
        results[metric] = dict(AP=float(evaluator.stats[0]), AP50=float(evaluator.stats[1]),
                               AP75=float(evaluator.stats[2]), per_class_AP50=per_class)
    return results


def iteration_limit(images, epochs, cap, batch_size, quick=False):
    return 2 if quick else min(cap, math.ceil(epochs * images / batch_size))


def select_batch(reports):
    """Use the slower GPU's throughput, requiring memory headroom and a 5% gain.

    No validation scores or held-out labels participate in this decision.
    """
    eligible = {}
    for batch, workers in reports.items():
        if len(workers) != 2 or any(
                r['seconds_per_iteration'] <= 0 or
                not math.isfinite(r['seconds_per_iteration']) or
                r['peak_reserved_gib'] > 0.80 * r['total_gpu_gib'] for r in workers):
            continue
        eligible[int(batch)] = min(int(batch) / r['seconds_per_iteration'] for r in workers)
    if not eligible:
        raise RuntimeError('No benchmarked batch has sufficient GPU memory headroom. '
                           'Try IMAGE_SIZE=512 with a new output directory.')
    chosen = max(eligible, key=eligible.get)
    if 1 in eligible and eligible[chosen] < 1.05 * eligible[1]:
        chosen = 1
    return chosen, eligible


def benchmark_batches(output, script, make_job, folds, requested, signature, quick):
    """Benchmark both GPUs concurrently, before allocating the training budget."""
    import concurrent.futures
    selection_path = output / 'batch_selection.json'
    previous = json.loads(selection_path.read_text()) if selection_path.exists() else None
    if previous and previous.get('signature') != signature:
        raise RuntimeError('Batch benchmark belongs to another experiment; use a new output.')
    candidates = ([previous['selected_batch_size']] if previous else
                  ([1] if quick and requested == 'auto' else
                   [1, 2] if requested == 'auto' else [int(requested)]))
    reports, rejected = {}, {}
    for batch in candidates:
        jobs = []
        for gpu in range(2):
            job = make_job(folds[0], 'preflight', 240)
            work = output / 'preflight' / f'batch_{batch}_gpu_{gpu}'
            work.mkdir(parents=True, exist_ok=True)
            sample = spread_sample(folds[0]['valid'], 12)
            job.update(work=str(work), batch_size=batch, max_iters=50,
                       validation_images=len(sample), valid_json=str(work / 'valid.json'))
            save_json(work / 'valid.json', coco_document(sample))
            save_json(work / 'job.json', job)
            jobs.append((gpu, work))
        errors = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            futures = []
            for gpu, work in jobs:
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                           OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
                futures.append((pool.submit(run_command,
                    [sys.executable, '-u', script, '--job', work / 'job.json'],
                    timeout=600, environment=env, log=str(work / 'worker.log')), work))
            for future, work in futures:
                try:
                    future.result()
                except subprocess.CalledProcessError as error:
                    log = (work / 'worker.log').read_text().lower()
                    if 'cuda out of memory' not in log and 'cuda error: out of memory' not in log:
                        raise
                    errors.append(str(error))
        if errors:
            rejected[str(batch)] = errors
        else:
            reports[batch] = [json.loads((work / 'preflight.json').read_text()) for _, work in jobs]
        for _, work in jobs:
            (work / 'final.pth').unlink(missing_ok=True)
    chosen, throughput = select_batch(reports)
    if previous and chosen != previous['selected_batch_size']:
        raise RuntimeError('Cannot change batch size when reusing completed folds.')
    selection = dict(signature=signature, selected_batch_size=chosen,
                     images_per_second_per_gpu=throughput, reports=reports, rejected=rejected,
                     policy='Slower GPU throughput; 20% memory headroom; 5% gain over batch 1')
    save_json(selection_path, selection)
    print('[batch selection]', json.dumps(selection), flush=True)
    return chosen, max(r['seconds_per_iteration'] for r in reports[chosen]), max(
        r['prediction_seconds_per_image'] for r in reports[chosen])


def plan_wave(remaining_seconds, pending_count, validation_images, step_seconds,
              prediction_seconds, iteration_cap):
    """Allocate one two-GPU wave after protecting final evaluation/export time."""
    # Protect pooled COCO scoring and packaging inside the run deadline.
    # Held-out inference uses the observed per-image speed, with a large margin.
    pooled_evaluation_reserve = 1200.
    packaging_reserve = 300.
    slot = (remaining_seconds - pooled_evaluation_reserve - packaging_reserve) / math.ceil(pending_count / 2)
    inference_reserve = max(420., validation_images * max(0.1, prediction_seconds) * 2.5 + 120)
    worker_overhead = 120.
    training_seconds = slot - inference_reserve - worker_overhead
    if training_seconds < 60:
        raise TimeoutError('Insufficient time for all remaining folds plus evaluation; no success claimed.')
    # Set a ceiling above the measured-speed projection so a fast iteration cap
    # does not waste the training slot. The wall-clock hook sets the real stop.
    iterations = min(iteration_cap, math.ceil(training_seconds / max(0.1, step_seconds * 0.9)))
    if iterations < 20:
        raise TimeoutError('Too little time remains for a meaningful training allocation.')
    return dict(slot_seconds=slot, train_seconds=training_seconds, max_iters=iterations,
                inference_reserve_seconds=inference_reserve,
                worker_overhead_seconds=worker_overhead,
                pooled_evaluation_reserve_seconds=pooled_evaluation_reserve,
                packaging_reserve_seconds=packaging_reserve)


def spread_sample(items, limit):
    """Benchmark image inference across the full validation set, not its first files."""
    if len(items) <= limit:
        return list(items)
    if limit < 2:
        raise ValueError('Benchmark sample needs at least two images')
    return [items[round(index * (len(items) - 1) / (limit - 1))]
            for index in range(limit)]


def bounded_fold_plan(shared_plan, current_plan, quick=False):
    """Give every fold the same planned compute unless the remaining time shrinks."""
    seconds = min(shared_plan['train_seconds'], current_plan['train_seconds'])
    if quick:
        seconds = min(seconds, 120)
    iterations = min(shared_plan['max_iters'], current_plan['max_iters'])
    return seconds, iterations


def write_evaluation_tables(output, state):
    """Only measured metrics; absent/incomplete metrics remain absent, never zero-filled."""
    rows = []
    for metric, values in (state.get('results') or {}).items():
        rows.append(dict(task=metric, scope='pooled OOF', AP=values['AP'],
                         AP50=values['AP50'], AP75=values['AP75']))
        for name, ap50 in values['per_class_AP50'].items():
            rows.append(dict(task=metric, scope=name, AP=None, AP50=ap50, AP75=None))
    with (output / 'evaluation_table.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['task', 'scope', 'AP', 'AP50', 'AP75'])
        writer.writeheader()
        writer.writerows(rows)
    def display(value):
        return '—' if value is None or value < 0 else f'{100 * value:.2f}%'
    lines = ['# Session 26 evaluation', '',
             'Complete five-fold OOF evaluation: ' + str(state.get('complete', False)),
             'Full budgeted training and evaluation complete: ' + str(
                 state.get('full_budgeted_training_and_evaluation_complete', False)),
             'All folds finished planned training: ' + str(
                 state.get('all_folds_finished_planned_training', False)),
             'All folds reached allocated target: ' + str(state.get('all_allocated_targets_reached', False)),
             'All folds reached requested iteration cap: ' + str(state.get('all_requested_caps_reached', False)), '',
             '| Task | Scope | AP50–95 | AP50 | AP75 |', '|---|---|---:|---:|---:|']
    for row in rows:
        lines.append(f"| {row['task']} | {row['scope']} | " +
                     ' | '.join(display(row[k]) for k in ('AP', 'AP50', 'AP75')) + ' |')
    if not rows:
        lines += ['', 'Evaluation not completed; no scores reported.']
    lines += ['', '| Fold | Status | Iterations / target | Effective epochs | Training hours | Inference minutes |',
              '|---|---|---:|---:|---:|---:|']
    fold_rows = []
    for f in range(5):
        marker = output / f'fold_{f}' / 'completed.json'
        report = json.loads(marker.read_text()) if marker.exists() else {}
        if report.get('signature') != state.get('signature'):
            report = {}
        status = ('planned time budget reached' if report.get('budget_stopped') else
                  'allocated target reached' if report.get('allocated_target_reached') else
                  'incomplete') if report else 'incomplete'
        fold_rows.append(dict(fold=f, status=status, **{k: report.get(k) for k in
            ('iterations', 'target_iterations', 'effective_epochs', 'train_seconds', 'evaluation_seconds')}))
        lines.append(f"| {f} | {status} | {report.get('iterations', '—')} / {report.get('target_iterations', '—')} | "
                     + (f"{report['effective_epochs']:.2f} | {report['train_seconds']/3600:.2f} | "
                        f"{report['evaluation_seconds']/60:.2f} |" if report else '— | — | — |'))
    with (output / 'fold_training_table.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(fold_rows[0]))
        writer.writeheader()
        writer.writerows(fold_rows)
    text = '\n'.join(lines) + '\n'
    (output / 'evaluation_table.md').write_text(text)
    print(text, flush=True)


def main(argv=None):
    arguments = list(sys.argv[1:] if argv is None else argv)
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-root', default='auto')
    parser.add_argument('--output', default='/kaggle/working/session26_v7')
    parser.add_argument('--hours', type=float, default=11.3)
    parser.add_argument('--size', type=int, choices=[512, 640, 768], default=640)
    parser.add_argument('--epochs', type=int, default=24)
    parser.add_argument('--batch-size', choices=['auto', '1', '2'], default='auto',
                        help='Benchmark batches 1 and 2 on both GPUs; select by images/second, never AP')
    parser.add_argument('--max-iters', type=int, default=60000,
                        help='Per-fold ceiling; the measured time budget sets a lower target if needed')
    parser.add_argument('--seed', type=int, default=2026)
    parser.add_argument('--quick', action='store_true', help='Two-iteration smoke test, NOT full OOF')
    parser.add_argument('--audit-only', action='store_true')
    parser.add_argument('--job', help=argparse.SUPPRESS)
    args = parser.parse_args(arguments)
    if args.job:
        worker(json.loads(Path(args.job).read_text()))
        return 0
    if not 0.5 <= args.hours <= 11.3 or args.epochs < 1 or args.max_iters < 20:
        parser.error('Use 0.5 <= hours <= 11.3, epochs >= 1 and max-iters >= 20')
    started = float(os.environ.get('SESSION26_STARTED', time.time()))
    if not args.audit_only:
        result = bootstrap(arguments, started)
        if result is not None:
            return result
    output = Path(args.output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    root = detect_dataset(args.data_root)
    items = read_items(root)
    folds, audit = make_folds(items, args.seed)
    assignment = {x['id']: f['fold'] for f in folds for x in f['valid']}
    manifest = [{k: x[k] for k in
        ('id', 'relative', 'split', 'group', 'pixel', 'label_sha256')} | dict(fold=assignment.get(x['id']))
        for x in items]
    signature = fingerprint(dict(build=BUILD, size=args.size, epochs=args.epochs, seed=args.seed,
                                 quick=args.quick, hours=args.hours, max_iters=args.max_iters,
                                 batch_size=args.batch_size, manifest=manifest))
    state_path = output / 'run_status.json'
    if state_path.exists() and json.loads(state_path.read_text()).get('signature') != signature:
        raise RuntimeError('Output contains a different experiment. Choose a new --output directory.')
    save_json(output / 'dataset_audit.json', audit)
    save_json(output / 'fold_manifest.json', manifest)
    for f in folds:
        for split in ('train', 'valid'):
            save_json(output / 'annotations' / f"fold_{f['fold']}_{split}.json", coco_document(f[split]))
    print(json.dumps(audit, indent=2), flush=True)
    if args.audit_only:
        return 0
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError('This time-budgeted configuration requires exactly two visible GPUs (T4 x2).')
    save_json(output / 'environment.json', dict(python=sys.version, executable=sys.executable,
        torch=torch.__version__, cuda=torch.version.cuda,
        gpus=[torch.cuda.get_device_name(i) for i in range(2)],
        packages=subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines()))
    state = dict(build=BUILD, signature=signature, complete=False, quick=args.quick, results=None)
    save_json(state_path, state)
    write_evaluation_tables(output, state)
    script = str(Path(__file__).resolve())
    initial = ENV_ROOT / 'initial_r101_point.pth'
    common = dict(size=args.size, seed=args.seed, initial=str(initial), signature=signature, batch_size=1)
    def make_job(f, action, seconds):
        index = f['fold']
        work = output / f'fold_{index}'
        work.mkdir(exist_ok=True)
        requested_max_iters = iteration_limit(len(f['train']), args.epochs, args.max_iters,
                                             common['batch_size'], args.quick)
        return dict(common, action=action, fold=index, work=str(work), train_seconds=seconds,
            max_iters=requested_max_iters, requested_max_iters=requested_max_iters,
            train_json=str(output / 'annotations' / f'fold_{index}_train.json'),
            valid_json=str(output / 'annotations' / f'fold_{index}_valid.json'),
            training_images=len(f['train']), validation_images=len(f['valid']))
    prepare = output / 'prepare_job.json'
    save_json(prepare, make_job(folds[0], 'prepare', 60))
    run_command([sys.executable, '-u', script, '--job', prepare], timeout=900)
    # Benchmark the actual two-worker load, including backward, save and inference.
    # OOM may reject a candidate; other errors must be fixed rather than hidden.
    chosen_batch, step_seconds, prediction_seconds = benchmark_batches(
        output, script, make_job, folds, args.batch_size, signature, args.quick)
    common['batch_size'] = chosen_batch
    prediction_seconds = max(0.1, prediction_seconds)
    state['training_configuration'] = dict(batch_size_per_fold=chosen_batch,
        learning_rate=0.00125 * chosen_batch, epoch_cap=args.epochs,
        iteration_cap=args.max_iters, hours=args.hours,
        checkpoint_selection='final time-budgeted checkpoint; no OOF selection')
    selected = folds[:1] if args.quick else folds
    pending = []
    for f in selected:
        marker = output / f"fold_{f['fold']}" / 'completed.json'
        artifacts = [marker.parent / name for name in ('final.pth', 'oof.bbox.json', 'oof.segm.json')]
        if marker.exists() and json.loads(marker.read_text()).get('signature') == signature and all(
                path.is_file() for path in artifacts):
            print('[resume] completed fold', f['fold'], flush=True)
        else:
            pending.append(f)
    deadline = started + args.hours * 3600
    # Freeze a common plan before wave one. Later waves may shrink it if the
    # measured workers slow down, but unused early inference time cannot make
    # later folds receive a larger training allocation than earlier folds.
    shared_plan = (plan_wave(deadline - time.time(), len(pending),
        max(len(f['valid']) for f in selected), step_seconds, prediction_seconds, args.max_iters)
        if pending else None)
    state['shared_training_plan'] = shared_plan
    save_json(state_path, state)
    try:
        while pending:
            allocation = plan_wave(deadline - time.time(), len(pending),
                max(len(f['valid']) for f in pending[:2]), step_seconds, prediction_seconds, args.max_iters)
            slot = allocation['slot_seconds']
            train_seconds, max_iters = bounded_fold_plan(shared_plan, allocation, args.quick)
            print('[budget]', json.dumps(allocation), flush=True)
            batch, pending = pending[:2], pending[2:]
            import concurrent.futures
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
                futures = []
                for gpu, f in enumerate(batch):
                    job = make_job(f, 'train', train_seconds)
                    job['max_iters'] = min(job['max_iters'], max_iters)
                    print(f"[fold {f['fold']}] GPU={gpu}, target={job['max_iters']} iterations, "
                          f"training limit={train_seconds/3600:.2f}h, held-out images={len(f['valid'])}", flush=True)
                    path = output / f"fold_{f['fold']}" / 'job.json'
                    save_json(path, job)
                    env = os.environ.copy()
                    env.update(CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
                    futures.append(pool.submit(run_command,
                        [sys.executable, '-u', script, '--job', path], timeout=slot,
                        environment=env, log=str(path.parent / 'worker.log')))
                for future in futures:
                    future.result()
            # Refine the next wave's plan using the slower completed worker.
            for f in batch:
                report = json.loads((output / f"fold_{f['fold']}" / 'completed.json').read_text())
                step_seconds = max(step_seconds, report['train_seconds'] / report['iterations'])
                prediction_seconds = max(prediction_seconds, report['evaluation_seconds'] / report['validation_images'])
            write_evaluation_tables(output, state)
        reports = [json.loads((output / f"fold_{f['fold']}" / 'completed.json').read_text()) for f in selected]
        state.update(folds=reports, elapsed_hours=(time.time()-started)/3600,
                     all_allocated_targets_reached=all(r['allocated_target_reached'] for r in reports),
                     all_requested_caps_reached=all(r['requested_cap_reached'] for r in reports),
                     all_folds_finished_planned_training=all(
                         r['allocated_target_reached'] or r['budget_stopped'] for r in reports))
        if not args.quick:
            development = [x for x in items if x['split'] != 'test']
            save_json(output / 'oof_ground_truth.json', coco_document(development))
            evaluation_job = output / 'evaluation_job.json'
            save_json(evaluation_job, dict(action='evaluate', output=str(output)))
            evaluation_seconds = deadline - time.time() - 300
            if evaluation_seconds < 60:
                raise TimeoutError('No safe time remains for pooled OOF evaluation and packaging.')
            run_command([sys.executable, '-u', script, '--job', evaluation_job],
                        timeout=evaluation_seconds, log=str(output / 'evaluation.log'))
            state['results'] = json.loads((output / 'evaluation_metrics.json').read_text())
            state['complete'] = True
            state['full_budgeted_training_and_evaluation_complete'] = bool(
                state['all_folds_finished_planned_training'])
            state['full_requested_training_and_evaluation_complete'] = bool(
                state['all_requested_caps_reached'])
        else:
            state['note'] = 'Smoke test only; not a completed five-fold training experiment.'
    except Exception as error:
        state['error'] = repr(error)
        raise
    finally:
        state['elapsed_hours'] = (time.time()-started)/3600
        write_evaluation_tables(output, state)
        save_json(state_path, state)
        # Metadata archive excludes multi-GB checkpoints; those remain separate Kaggle outputs.
        with zipfile.ZipFile(output.parent / (output.name + '_reports.zip'), 'w', zipfile.ZIP_DEFLATED) as archive:
            for path in output.rglob('*'):
                if path.is_file() and path.suffix in {'.json', '.log', '.py', '.csv', '.md'}:
                    archive.write(path, path.relative_to(output))
    print(json.dumps(state, indent=2), flush=True)
    return 0


if __name__ == '__main__':
    raise SystemExit(main([] if 'ipykernel' in sys.modules else None))


+ /opt/bin/nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
+ /usr/bin/python3 -m pip install --no-cache-dir --target /tmp/session26_openmmlab/installer uv==0.8.22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 233.5 MB/s eta 0:00:00
+ /tmp/session26_openmmlab/installer/bin/uv venv --python 3.11 --seed /tmp/session26_openmmlab/venv --allow-existing


Using CPython 3.11.13
Creating virtual environment with seed packages at: /tmp/session26_openmmlab/venv
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /tmp/session26_openmmlab/venv/bin/activate


+ /tmp/session26_openmmlab/venv/bin/python -m pip install --no-cache-dir --timeout 90 pip==25.2 setuptools==75.1.0 wheel==0.44.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 322.3 MB/s  0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.48.0
    Uninstalling wheel-0.48.0:
      Successfully uninstalled wheel-0.48.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0
  Attempting uninstall: pip
    Found existing installation: pip 26.2.1
    Uninstalling pip-26.2.1:
      Successfully uninstalled pip-26.2.1

+ /tmp/session26_openmmlab/venv/bin/python -m pip install --no-cache-dir --timeout 90 torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
